# Customer Churn Analysis & Retention Dashboard

## Project Objective

This project aims to analyze customer churn behavior, identify key churn drivers, segment customers based on risk levels, and generate actionable retention recommendations using Python, SQL, and Power BI.

# 01 Data Understanding

## Objective

Understand the dataset structure, validate data quality, identify missing values, and prepare the data for analysis.

In [1]:
%pip install seaborn
%pip install openpyxl

In [2]:
# ============================
# Import Libraries
# ============================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [3]:
# ============================
# Load Dataset
# ============================

df = pd.read_excel(
    "/drive/customer-churn-analytics-project/data/Telco_customer_churn.xlsx"
)

df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


# 01 Data Understanding

## Objective

The purpose of this phase is to understand the dataset structure, validate data quality, identify missing values, detect duplicates, and assess data types before cleaning and analysis.

In [4]:
# Dataset Shape

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 7043
Columns: 33


In [5]:
# Dataset Information

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [6]:
# Missing Values

df.isnull().sum().sort_values(ascending=False)

Churn Reason         5174
Online Security         0
CLTV                    0
Churn Score             0
Churn Value             0
Churn Label             0
Total Charges           0
Monthly Charges         0
Payment Method          0
Paperless Billing       0
Contract                0
Streaming Movies        0
Streaming TV            0
Tech Support            0
Device Protection       0
Online Backup           0
CustomerID              0
Count                   0
Multiple Lines          0
Phone Service           0
Tenure Months           0
Dependents              0
Partner                 0
Senior Citizen          0
Gender                  0
Longitude               0
Latitude                0
Lat Long                0
Zip Code                0
City                    0
State                   0
Country                 0
Internet Service        0
dtype: int64

In [7]:
# Duplicate Records

print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [8]:
# Column Names

df.columns.tolist()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

# Initial Findings

### Key Observations

- The dataset contains 7,043 customer records and 33 features.
- No duplicate records were identified.
- Most variables contain complete information.
- Missing values are concentrated in the **Churn Reason** column, which is expected because churn reasons are only available for customers who left the company.
- The **Total Charges** column appears to have an incorrect data type and requires further validation.

# 02 Data Cleaning

## Objective

Prepare the dataset for analysis by correcting data types, handling missing values, and removing irrelevant columns that do not contribute to churn analysis.

In [9]:
# Check Total Charges

df["Total Charges"].head()

0     108.15
1     151.65
2      820.5
3    3046.05
4     5036.3
Name: Total Charges, dtype: object

In [10]:
# Current Data Type

df["Total Charges"].dtype

dtype('O')

In [11]:
# Convert Total Charges to Numeric

df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

df["Total Charges"].dtype

dtype('float64')

In [12]:
# Missing Values After Conversion

df["Total Charges"].isnull().sum()

np.int64(11)

In [13]:
# Remove Missing Values

df = df.dropna(subset=["Total Charges"])

print(df.shape)

(7032, 33)


In [14]:
# Remove Irrelevant Columns

drop_cols = [
    "CustomerID",
    "Count",
    "Country",
    "State",
    "Lat Long",
    "Latitude",
    "Longitude"
]

df = df.drop(columns=drop_cols)

df.shape

(7032, 26)

In [15]:
# Final Dataset Information

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   City               7032 non-null   object 
 1   Zip Code           7032 non-null   int64  
 2   Gender             7032 non-null   object 
 3   Senior Citizen     7032 non-null   object 
 4   Partner            7032 non-null   object 
 5   Dependents         7032 non-null   object 
 6   Tenure Months      7032 non-null   int64  
 7   Phone Service      7032 non-null   object 
 8   Multiple Lines     7032 non-null   object 
 9   Internet Service   7032 non-null   object 
 10  Online Security    7032 non-null   object 
 11  Online Backup      7032 non-null   object 
 12  Device Protection  7032 non-null   object 
 13  Tech Support       7032 non-null   object 
 14  Streaming TV       7032 non-null   object 
 15  Streaming Movies   7032 non-null   object 
 16  Contract           7032 non-n

# Data Cleaning Summary

### Actions Performed

- Converted **Total Charges** from object to numeric format.
- Removed records with missing Total Charges values.
- Removed non-analytical identifiers and geographic coordinate fields.
- Retained customer demographic, service, billing, and churn-related variables for analysis.

### Outcome

The dataset is now cleaned and ready for exploratory data analysis (EDA).

In [16]:
# Save Clean Dataset

df.to_csv(
    "/drive/customer-churn-analytics-project/data/customer_churn_cleaned.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


# Conclusion

The dataset has been successfully cleaned and prepared for analysis.

Key preprocessing steps included:

- Data quality assessment
- Missing value handling
- Data type correction
- Removal of non-analytical features
- Exporting a clean dataset for subsequent analysis

The cleaned dataset is now ready for exploratory data analysis (EDA).